# Exposure-response model choice — the dose-extrapolation model-selection axis

The TGI→survival chain begins with one upstream choice the rest of Onkos takes as given: the
**exposure-response (ER) model** that maps a drug exposure to the effect magnitude driving the kill
term. Emax, sigmoid-Emax, and power all fit comparably at the studied dose — and *diverge* when you
extrapolate to a dose the trial did not study, which is exactly what dose selection asks them to do.

This is the project's **transportability thesis applied one layer upstream**: anchor the shapes to
agree at the studied dose, then watch the predicted effect (and OS) diverge off it. *Population/trial
level only; the analysis re-anchors the curated ER shapes, it does not refit them, and it ranks no doses.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.dose_response import calibrated_er, compare_er_extrapolation, ER_SHAPE_RECORDS

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
C_REF, E_REF = 150.0, 1.0   # the studied dose and the effect all shapes share there

## All shapes agree at the studied dose — by construction

Each ER shape keeps its curated shape parameters (EC50, gamma, theta) but has its scale re-solved so
it passes through `(C_ref, E_ref)`. So at the studied dose they are indistinguishable; the only thing
that differs is how they extrapolate.

In [ ]:
for er in ER_SHAPE_RECORDS:
    f = calibrated_er(ds, er, c_ref=C_REF, e_ref=E_REF)
    name = er.split('.')[-1]
    print(f'{name:24s}  E(studied)={f(C_REF):.3f}   E(half)={f(0.5*C_REF):.3f}   E(double)={f(2*C_REF):.3f}')

## The OS consequence: zero at the anchor, growing on extrapolation

Feed each shape's effect through the existing TGI→survival chain. At the studied dose the ER-model
choice has **no** OS consequence (the control). Off it, the choice moves the OS prediction — most on
**de-escalation**, where the effect sits on the steep part of the survival relationship.

In [ ]:
cmp = compare_er_extrapolation(ds, 'resistance.claret_2009.tgi', context=ctx, c_ref=C_REF, e_ref=E_REF)
print(f"{'dose':>6} {'effect spread':>14} {'OS spread (wk)':>15}")
for r in cmp.rows:
    tag = '  <- studied dose' if abs(r['dose']-C_REF) < 1e-9 else ''
    print(f"{r['dose']:>6.0f} {r['effect_divergence']:>14.2f} {r['os_divergence']:>15.0f}{tag}")
print()
print(f'OS spread at the studied dose = {cmp.reference_os_divergence:.1f} wk (anchored control)')
print(f'max OS spread on extrapolation = {cmp.max_os_divergence:.0f} wk')
print(f'de-escalation (quarter dose) = {cmp.os_divergence_at(0.25*C_REF):.0f} wk  vs  '
      f'escalation (4x dose) = {cmp.os_divergence_at(4*C_REF):.0f} wk')

## The takeaway

A dose-response model that fits the studied dose perfectly still carries an unquantified
model-selection risk the moment it is used to choose a *different* dose — and the risk is sharpest for
de-escalation, the very question (can we lower the dose?) that ER models are built to answer. Onkos
makes the ER-shape choice explicit and quantifies its OS consequence as a function of how far you
extrapolate. **No dose recommendation, no individual prediction; the underlying model's tier governs
and cannot be raised.**